In [ ]:
import pandas as pd
from dotenv import load_dotenv

True

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

In [ ]:
model = init_chat_model("openai:gpt-5.6-luna", reasoning_effort="none")

### 시나리오 설정
- 질문에 대한 답변을 사투리로 바꿔서 결과를 마크다운으로 저장하게 하기
1. 질문에 대한 답변을 한다
2. 답변을 나만의 사투리로 변환한다
3. 결과를 마크다운으로 저장한다

In [4]:
# 나만의 사투리로 변환하는 툴
@tool
def change_text(question : str):
    """답변을 받아서 나만의 사투리로 변환한다."""

    result = "나만의 사투리" + question + "변환했어요"

    return result

In [5]:
# 결과를 md로 저장하는 툴
@tool
def save_md(text : str):
    """최종 결과를 받아서 마크다운 문서로 저장한다.
    파일명은 result.md
    """

    with open("result.md", "w", encoding="utf-8-sig") as file:
        file.write(text)

    return "저장 완료 : result.md"  # LLM에게 갈거기 때문에 이런식으로

In [6]:
agent = create_agent(
    model=model,
    tools=[change_text,save_md],
    system_prompt="""너는 사용자가 질문을 하면 답변하고, 적절하게 도구를 호출하여 답변하는 비서야."""
)

In [7]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# agent 호출용 함수
def ask(question):
    response = agent.invoke({"messages" : [ HumanMessage(question) ]})
    return response

In [8]:
result = ask("안녕하세요")
result['messages']

[HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='8943e35e-0d05-4627-9115-f84b58c3dad8'),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 200, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ER7ouUDtnEnuCO1I1IprLsfN7FDFZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0cc47-551f-7d70-84eb-0361e6bdafae-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 200, 'output_tokens': 14, 'total_tokens': 214, 'input_token_details': {'audio

In [ ]:
import os

load_dotenv()

True

In [ ]:
# os.environ["TAVILY_API_KEY"] # 확인용.

### 웹서치 가능하게 하기

In [13]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_result=5,
    topic="general"
)

In [ ]:
search_tool.invoke({"query" : "오늘 서울의 날씨는 어때?"})

### tool에 웹서치 넣어보기

In [18]:
search_agent = create_agent(
    model=model,
    tools=[search_tool],
    system_prompt="사용자의 질문에 적합한 도구를 호출해서 답변해. 필요하지 않으면 호출하지 말고"
)

In [19]:
def ask(question):
    response = search_agent.invoke({"messages" : [ HumanMessage(question) ]})
    return response

result = ask("서울의 2026년 9월 23일의 날씨는?")
result['messages']

[HumanMessage(content='서울의 2026년 9월 23일의 날씨는?', additional_kwargs={}, response_metadata={}, id='313a2fd6-ff46-41eb-930f-4fb23e33fbbe'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 1313, 'total_tokens': 1346, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 1310, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ER88jfqW8Tnx2IvIk0bAT8D0xIZLx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0cc5a-1fc9-7b92-ae69-497aeb969f68-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '서울 2026년 9월 23일 날씨 예보'}, 'id': 'call_Je8kO5U1VVQj9mq7EQSU3AoQ', 'type': 'tool_ca